## Feature Extraction – Training Set

In [ ]:
import os, glob
import numpy as np
from astropy.io import fits
from scipy import ndimage
from scipy.ndimage import gaussian_filter
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def validate_data(data, fname):
    if data is None: raise ValueError("Data is None")
    if data.ndim != 2: raise ValueError(f"Expected 2D, got {data.ndim}D")
    if data.shape[0] < 10 or data.shape[1] < 10:
        raise ValueError(f"Too small: {data.shape}")
    if np.all(data == 0): raise ValueError("All zeros")
    return True

def create_rings(data, n=10):
    cy, cx = data.shape[0]//2, data.shape[1]//2
    y, x   = np.ogrid[:data.shape[0], :data.shape[1]]
    dist   = np.maximum(np.abs(x - cx), np.abs(y - cy))
    max_d  = min(cx, cy, data.shape[0]-cy, data.shape[1]-cx)
    w = max_d / n
    return [(dist >= i*w) & (dist < (i+1)*w) for i in range(n)]

def calc_DA(data, rings):
    vals = [np.mean(data[r]) if np.any(r) else 0 for r in rings]
    v = np.array(vals[2:])
    if len(v) == 0 or v.max() == 0: return 0
    v /= v.max()
    return np.trapz(v) / len(v)

def calc_DV(data, rings):
    vals = [np.var(data[r]) if np.any(r) and r.sum() > 1 else 0 for r in rings]
    v = np.array(vals)
    if v.max() > 0: v /= v.max()
    return v.mean()

def calc_asymmetry(data):
    try:
        rot = ndimage.rotate(data, 180, reshape=False, order=1)
        tot = np.sum(np.abs(data))
        return np.sum(np.abs(data - rot)) / tot if tot > 0 else 0
    except: return 0

def calc_clumpiness(data):
    try:
        sm = ndimage.gaussian_filter(data, sigma=7)
        diff = np.clip(data - sm, 0, None)
        diff[data >= np.percentile(data, 99)] = 0
        tot = np.sum(np.abs(data))
        return np.sum(diff) / tot if tot > 0 else 0
    except: return 0

def calc_concentration(data):
    try:
        cy, cx = data.shape[0]//2, data.shape[1]//2
        y, x   = np.ogrid[:data.shape[0], :data.shape[1]]
        r      = np.sqrt((x-cx)**2 + (y-cy)**2)
        total  = np.sum(data)
        if total == 0: return 0, 0, 0
        idx = np.argsort(r.flatten())
        cs  = np.cumsum(data.flatten()[idx]) / total
        i50 = np.searchsorted(cs, 0.5)
        i90 = np.searchsorted(cs, 0.9)
        r50 = r.flatten()[idx[i50]] if i50 < len(idx) else 0
        r90 = r.flatten()[idx[i90]] if i90 < len(idx) else 0
        return (r50/r90 if r90 > 0 else 0), r50, r90
    except: return 0, 0, 0

def calc_bar_bulge_ratio(data):
    try:
        cy, cx = data.shape[0]//2, data.shape[1]//2
        h, w   = data.shape
        ys, ye = max(0,int(cy-h*0.2)), min(h,int(cy+h*0.2))
        xs, xe = max(0,int(cx-w*0.2)), min(w,int(cx+w*0.2))
        c = data[ys:ye, xs:xe]
        if c.size == 0 or c.shape[0] < 5: return 0
        yg, xg  = np.mgrid[:c.shape[0], :c.shape[1]]
        tot = c.sum()
        if tot == 0: return 0
        xm = (xg*c).sum()/tot; ym = (yg*c).sum()/tot
        mxx = (c*(xg-xm)**2).sum()/tot
        myy = (c*(yg-ym)**2).sum()/tot
        mxy = (c*(xg-xm)*(yg-ym)).sum()/tot
        tr, det = mxx+myy, mxx*myy - mxy**2
        if det > 0 and tr > 0:
            disc = tr**2 - 4*det
            if disc >= 0:
                lp = 0.5*(tr + np.sqrt(disc)); lm = 0.5*(tr - np.sqrt(disc))
                return np.sqrt(lp/lm) if lm > 0 else 0
        return 0
    except: return 0

def calc_central_concentration(data):
    try:
        cy, cx = data.shape[0]//2, data.shape[1]//2
        y, x   = np.ogrid[:data.shape[0], :data.shape[1]]
        r      = np.sqrt((x-cx)**2 + (y-cy)**2)
        ir     = min(cx,cy) * 0.1
        inner  = data[r <= ir].sum(); outer = data[r > ir].sum()
        return inner/outer if outer > 0 else 0
    except: return 0

def calc_spiral_arm_strength(data):
    try:
        ss = gaussian_filter(data, sigma=3); sl = gaussian_filter(data, sigma=10)
        arm = np.clip(ss - sl, 0, None)
        tot = data.sum()
        return arm.sum()/tot if tot > 0 else 0
    except: return 0

def calc_gini(data):
    try:
        flat = data.flatten(); flat = flat[flat > 0]
        if len(flat) == 0: return 0
        s = np.sort(flat); n = len(s); idx = np.arange(1, n+1)
        return (2*np.sum(idx*s))/(n*np.sum(s)) - (n+1)/n
    except: return 0

def calc_ellipticity(data):
    try:
        cy, cx = data.shape[0]//2, data.shape[1]//2
        y, x   = np.ogrid[:data.shape[0], :data.shape[1]]
        tot    = data.sum()
        if tot == 0: return 0
        mxx = (data*(x-cx)**2).sum()/tot
        myy = (data*(y-cy)**2).sum()/tot
        mxy = (data*(x-cx)*(y-cy)).sum()/tot
        tr, det = mxx+myy, mxx*myy - mxy**2
        return (1 - np.sqrt(4*det/tr**2)) if (det > 0 and tr > 0) else 0
    except: return 0

def calc_petrosian(data):
    try:
        cy, cx = data.shape[0]//2, data.shape[1]//2
        y, x   = np.ogrid[:data.shape[0], :data.shape[1]]
        r      = np.sqrt((x-cx)**2 + (y-cy)**2)
        mr     = min(cx, cy)
        if mr < 10: return 0
        radii  = np.linspace(0, mr, 50)
        etas   = []
        for radius in radii[1:]:
            ann = (r >= radius - mr/50) & (r < radius)
            circ= r < radius
            if np.any(ann) and np.any(circ):
                ab = data[ann].mean(); cb = data[circ].mean()
                etas.append(ab/cb if cb > 0 else 0)
            else:
                etas.append(0)
        etas = np.array(etas)
        if len(etas) > 0 and np.any(etas < 0.2):
            return radii[np.where(etas < 0.2)[0][0]+1] / mr
        return 1.0
    except: return 0

input_folder  = "train"
output_folder = "featurestrain"
os.makedirs(output_folder, exist_ok=True)


In [ ]:
# Extra features specific to train notebook (color proxy, gas conc, etc.)
def calc_color_proxy(data):
    try:
        lap = ndimage.laplace(data)
        tex = np.var(lap)
        return np.tanh(tex / (np.mean(data) + 1e-6) * 0.5)
    except: return 0

def calc_gas_conc(data):
    try:
        cy, cx = data.shape[0]//2, data.shape[1]//2
        y, x   = np.ogrid[:data.shape[0], :data.shape[1]]
        r_norm = np.sqrt((x-cx)**2 + (y-cy)**2) / np.sqrt(cx**2+cy**2)
        tot    = data.sum()
        return data[r_norm <= 0.3].sum() / tot if tot > 0 else 0
    except: return 0

def calc_radial_slope(data):
    try:
        cy, cx = data.shape[0]//2, data.shape[1]//2
        y, x   = np.ogrid[:data.shape[0], :data.shape[1]]
        r      = np.sqrt((x-cx)**2 + (y-cy)**2).flatten()
        d      = data.flatten()
        bins   = np.linspace(0, r.max(), 30)
        means  = ndimage.mean(d, labels=np.digitize(r, bins), index=np.arange(1, len(bins)+1))
        valid  = means > 0
        if valid.sum() < 5: return 0
        slope, _ = np.polyfit(np.log10(bins[valid]+1e-6), np.log10(means[valid]+1e-6), 1)
        return slope
    except: return 0

def calc_bar_fourier(data):
    try:
        cy, cx = data.shape[0]//2, data.shape[1]//2
        y, x   = np.indices(data.shape)
        r      = np.sqrt((x-cx)**2 + (y-cy)**2)
        theta  = np.arctan2(y-cy, x-cx)
        rb     = np.linspace(0, r.max(), 20)
        a2s    = []
        for i in range(len(rb)-1):
            ann = (r >= rb[i]) & (r < rb[i+1])
            if np.any(ann):
                I = data[ann]; phi = theta[ann]
                a2 = np.abs(np.sum(I * np.exp(2j*phi))) / (np.sum(I)+1e-6)
                a2s.append(a2)
        return np.mean(a2s) if a2s else 0
    except: return 0

def calc_spiral_pitch(data):
    try:
        gy, gx = np.gradient(gaussian_filter(data, 2))
        angles = np.degrees(np.arctan2(gy, gx))
        valid  = data > np.percentile(data, 70)
        return np.tanh(np.std(angles[valid]) / 60)
    except: return 0

def calc_bulge_to_total(data):
    try:
        cy, cx = data.shape[0]//2, data.shape[1]//2
        y, x   = np.ogrid[:data.shape[0], :data.shape[1]]
        r_n    = np.sqrt((x-cx)**2 + (y-cy)**2) / np.sqrt(cx**2+cy**2)
        tot    = data.sum()
        return data[r_n <= 0.2].sum() / tot if tot > 0 else 0
    except: return 0

def calc_ang_momentum(data):
    try:
        cy, cx = data.shape[0]//2, data.shape[1]//2
        q1 = data[:cy, :cx]; q3 = data[cy:, cx:]
        if q1.size == 0 or q3.size == 0: return 0
        q1n = (q1 - q1.mean()) / (q1.std() + 1e-6)
        q3n = (q3 - q3.mean()) / (q3.std() + 1e-6)
        q3r = ndimage.zoom(q3n, (q1.shape[0]/q3.shape[0], q1.shape[1]/q3.shape[1]))
        return np.mean(q1n * q3r)
    except: return 0


In [ ]:
fits_files = glob.glob(os.path.join(input_folder, "*.fits"))
total = len(fits_files)
print(f"Extracting features from {total} training files")
print("=" * 60)

records, errors = [], 0

for i, fpath in enumerate(fits_files, 1):
    fname = os.path.basename(fpath)
    if i % max(1, total//10) == 0:
        print(f"Progress: {i}/{total} ({100*i//total}%)  errors={errors}")
    try:
        with fits.open(fpath) as hdul:
            data = hdul[0].data
        validate_data(data, fname)
        if data.max() > 0:
            data = data.astype(float) / data.max()
        else:
            raise ValueError("max=0")
        rings = create_rings(data)
        a180, a90 = calc_asymmetry(data), calc_asymmetry(data)
        conc, r50, r90 = calc_concentration(data)
        records.append({
            'filename':               fname,
            'DA':                     calc_DA(data, rings),
            'DV':                     calc_DV(data, rings),
            'Asymmetry_180':          a180,
            'Asymmetry_90':           a90,
            'Clumpiness':             calc_clumpiness(data),
            'Concentration':          conc,
            'R50':                    r50,
            'R90':                    r90,
            'Bar_Bulge_Ratio':        calc_bar_bulge_ratio(data),
            'Central_Concentration':  calc_central_concentration(data),
            'Spiral_Arm_Strength':    calc_spiral_arm_strength(data),
            'Gini':                   calc_gini(data),
            'Ellipticity':            calc_ellipticity(data),
            'Petrosian_Radius':       calc_petrosian(data),
            'Color_Proxy':            calc_color_proxy(data),
            'Gas_Concentration':      calc_gas_conc(data),
            'Radial_Slope':           calc_radial_slope(data),
            'Bar_Fourier_Strength':   calc_bar_fourier(data),
            'Spiral_Pitch_Angle':     calc_spiral_pitch(data),
            'Bulge_To_Total':         calc_bulge_to_total(data),
            'Angular_Momentum_Proxy': calc_ang_momentum(data),
        })
    except Exception as e:
        errors += 1
        if errors <= 10:
            print(f"  ERROR: {fname} – {str(e)[:80]}")

df = pd.DataFrame(records)
csv_path = os.path.join(output_folder, "galaxy_features_enhanced.csv")
df.to_csv(csv_path, index=False)
print(f"\n{'='*60}")
print(f"Done! Extracted {len(df)} vectors, {errors} errors")
print(f"Features: {len(df.columns)-1}")
print(f"Saved to: {csv_path}")
print(df.describe())
